# RHO-FFT · P1（Colab）：一阶供给的测量，H1 与 H1a

这个 notebook 在一次会话里完成两件事：先确认基线没有被改坏（P0 的回归检查），再跑 P1 的消融阶梯，给出 H1（零阶更新有多少信息）和 H1a（偏向采样的收益是不是来自稀释）的第一批证据。全部逻辑在仓库的 `experiments/p0/p0.py` 和 `experiments/p1/p1.py` 里。

**运行前**

1. 运行时类型选 GPU：默认 G4，模型较大时可换 H100。
2. 代码只从 GitHub 读取（`GuangLun2000/RHO-FFT`，分支见参数 `REPO_REF`）。仓库私有时在 Colab 的"密钥"里添加 `GITHUB_TOKEN`。
3. 不使用 Google Drive。结果留在本页面的输出里，每个阶段结束时浏览器会下载一个结果压缩包（浏览器可能询问是否允许多次下载，请允许）。会话之间不保存任何东西：没跑完的运行下次要重跑。
4. 从上到下运行全部单元格。

**步骤一览**

| 步骤 | 内容 | 用不用 GPU |
|---|---|---|
| A1 到 A3 | 会话、取代码、建环境 | 不用 |
| B | 预演：小模型把 P0 回归和 P1 全流程走一遍，约四分钟 | 不用 |
| C1 | 环境记录与单元测试 | 探测一次 |
| C2 | 黄金值与基线回归（I1、I2） | 用，规模很小 |
| C3 | E1a 供给图（不训练） | 不用 |
| C4 | 第 1 组：判定 H1a 的 5 次运行 | 用 |
| C5 | 第 2 组：H1 的 2 次运行，加 FedAvg、DeComFL 两个参照 | 用 |
| C6 | 第 3 组：阶梯里剩下的 3 次运行 | 用 |
| D | 总反馈与曲线 | 不用 |
| E | 状态汇总，下载结果，释放 GPU | |

**实验臂（Qwen2.5-1.5B / AG News，Dirichlet 0.5，160 轮，其余均为 HO-FL 默认超参数）**

| 臂 | 设置 | 它回答什么 |
|---|---|---|
| A | HO-FL 原样（β=0 和 β=0.75） | 基线；同时记录逐层统计，并在第 1、20、40、80、160 轮运行反事实一阶探针 |
| B | 关闭零阶更新，聚合权重仍为 1/K | A 减 B 是零阶更新的净贡献（H1） |
| C | 每层只在一阶报告之间取平均（`fo_mean`） | C 减 A 是去掉稀释的收益（H1a） |
| D | 关闭零阶更新，且只在一阶报告之间平均 | 只做部分层一阶训练能到哪里 |
| A×2、A×3 | 基线，服务器步长乘 2、乘 3（仅 β=0） | 只放大步长能否复现 C 的收益 |
| FedAvg、DeComFL | 全一阶、全零阶 | 上界与下界 |

**时间预算与并行**

`BUDGET_HOURS` 是整个 notebook 的总预算。每种成本类型先单独跑 3 轮预检，测出每轮耗时、每次评估耗时和显存峰值；之后按优先级排队，估计放不进剩余预算的运行会被跳过并在报告里列出。`JOBS` 是允许同时共用 GPU 的运行数：每个运行是独立进程，数值与单独运行相同；实际并行数还会按预检测到的显存峰值自动下调。并行能快多少没有实测过，所以在测到之前按"并行不带来任何加速"来估计时间。

In [ ]:
#@title Parameters { display-mode: "form" }
# The code is read from GitHub only.
REPO_URL = "https://github.com/GuangLun2000/RHO-FFT.git"  #@param {type:"string"}
REPO_REF = "p1-measurement"  #@param {type:"string"}
ASK_FOR_TOKEN = True  #@param {type:"boolean"}

# Total wall-clock budget of this notebook, in hours.
BUDGET_HOURS = 5.5  #@param {type:"number"}

# "venv": Python 3.11 with the pinned dependencies of pyproject.toml (recommended).
# "system": Colab's own Python and torch; only a fallback, versions then differ from the pins.
ENV_MODE = "venv"  #@param ["venv", "system"]

# What to run.
MODEL = "qwen"  #@param ["qwen", "opt"]
SEEDS = "42"  #@param {type:"string"}
# Runs that may share the GPU. Each run is its own process; the number is lowered automatically if the measured
# memory of one run does not allow it. Use 1 to run strictly one after the other.
JOBS = 2  #@param {type:"integer"}

# Stages.
RUN_TINY_DRY_RUN = True  #@param {type:"boolean"}
RUN_TESTS = True  #@param {type:"boolean"}
RUN_REGRESSION = True  #@param {type:"boolean"}
RUN_BLOCK_1 = True  #@param {type:"boolean"}
RUN_BLOCK_2 = True  #@param {type:"boolean"}
RUN_BLOCK_3 = True  #@param {type:"boolean"}
# Run the experiment blocks even when the regression check did not pass (not recommended).
FORCE = False  #@param {type:"boolean"}
# Download the result bundle through the browser after every block and before the runtime is released.
DOWNLOAD_BUNDLE = True  #@param {type:"boolean"}
# Release the GPU runtime after the last cell, and after an error in any cell.
RELEASE_GPU = True  #@param {type:"boolean"}

## A1. 会话与 GPU

记录开始时间（时间预算从这里算起），显示 GPU，注册"出错后先保存结果再释放 GPU"的处理器。

In [ ]:
#@title A1 · Session and GPU
import base64
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

SESSION_START = time.time()
DEADLINE = SESSION_START + 3600 * BUDGET_HOURS
REPO_DIR = "/content/RHO-FFT"
WORK = "/content/work/p1"            # results of the P1 pipeline (local disk, fast)
P0_WORK = "/content/work/p0"         # results of the baseline regression (golden values, verdicts)
TINY_WORK = "/content/work/p1-tiny"  # results of the CPU dry run, kept apart from the real ones
TINY_P0_WORK = "/content/work/p0-tiny"
CACHE = "/content/cache"             # models, datasets, token caches and manifests
VENV = "/content/venv"


def sh(command, cwd=None, env=None, check=True):
    # Run a command, stream its output into the cell, and stop on failure when check is set.
    shown = command if isinstance(command, str) else " ".join(str(part) for part in command)
    print(f"$ {shown}", flush=True)
    process = subprocess.Popen(command, shell=isinstance(command, str), cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {shown}")
    return code


def hours_left():
    return (DEADLINE - time.time()) / 3600


STATUS = {}  # one line per stage, printed in the last cell


def release_gpu(delay=10):
    # Hand the GPU runtime back. Everything on the VM's local disk is lost, so this runs only after the results
    # have been shown and downloaded.
    if not RELEASE_GPU:
        print("RELEASE_GPU is off; the runtime stays connected.")
        return
    print(f"Releasing the GPU runtime in {delay} seconds ...", flush=True)
    time.sleep(delay)
    from google.colab import runtime
    runtime.unassign()


def stop(message):
    # A failed check stops "Run all"; the error handler below then releases the GPU.
    raise RuntimeError(message)


def download_results(refresh=True):
    # Send the result bundle to the browser. With refresh, the feedback is rebuilt first from whatever exists, so
    # that an interrupted session still leaves with its finished runs. Returns whether a download was started.
    if refresh and "p1" in globals() and Path(WORK).exists():
        try:
            p1("feedback", extra=["--quiet"])
        except Exception as error:
            print("feedback could not be rebuilt:", repr(error))
    bundle = Path(WORK) / "feedback" / "feedback_P1_bundle.zip"
    if not (DOWNLOAD_BUNDLE and bundle.exists()):
        return False
    try:
        from google.colab import files
        files.download(str(bundle))
        return True
    except Exception as error:
        print("download failed:", repr(error))
        return False


def _release_on_error(shell, etype, value, traceback, tb_offset=None):
    # Any error in any later cell: show it, save what exists, then release the GPU instead of leaving it idle.
    shell.showtraceback((etype, value, traceback), tb_offset=tb_offset)
    download_results()
    release_gpu(30 if DOWNLOAD_BUNDLE else 10)


# Manual interruption (KeyboardInterrupt) is not an Exception and does not release the runtime.
get_ipython().set_custom_exc((Exception,), _release_on_error)


sh("nvidia-smi", check=False)

## A2. 取代码

克隆仓库并检出 `REPO_REF`。回归检查要用 `git archive` 取出两个参照提交（上游原样导入、插桩提交），所以需要带完整历史的克隆。token 只通过环境变量传给 git，不会出现在输出里。

In [ ]:
#@title A2 · Get the code
def git_environment():
    # GitHub credentials for a private repository, passed to git through the environment only.
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as error:
        print("Colab secret GITHUB_TOKEN not available:", type(error).__name__)
    if not token and ASK_FOR_TOKEN:
        from getpass import getpass
        token = getpass("GitHub token with read access (leave empty for a public repository): ").strip()
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    return env


source, git_env = REPO_URL, git_environment()

if not os.path.isdir(f"{REPO_DIR}/.git"):
    sh(["git", "clone", "--quiet", source, REPO_DIR], env=git_env)
else:
    sh(["git", "-C", REPO_DIR, "fetch", "--quiet", "origin"], env=git_env)
remote_ref = f"origin/{REPO_REF}"
has_remote_ref = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", remote_ref],
                                capture_output=True).returncode == 0
sh(["git", "-C", REPO_DIR, "checkout", "--quiet", "--detach", remote_ref if has_remote_ref else REPO_REF])
sh(["git", "-C", REPO_DIR, "log", "--oneline", "-8"])

revisions = json.loads(Path(f"{REPO_DIR}/experiments/p0/revisions.json").read_text())
for name, sha in revisions.items():
    if subprocess.run(["git", "-C", REPO_DIR, "cat-file", "-e", sha + "^{commit}"], capture_output=True).returncode != 0:
        raise RuntimeError(f"Reference revision {name} ({sha}) is missing from this clone; a full-history clone is required.")
print("reference revisions present:", revisions)

## A3. 环境

建一个独立的 Python 3.11 虚拟环境，按 `pyproject.toml` 的固定版本安装依赖（PyTorch 2.6.0 / CUDA 12.4）。所有训练都在这个环境的子进程里运行。仓库本身不安装，运行时通过 `PYTHONPATH` 指向源码树。

In [ ]:
#@title A3 · Environment
def clean_environment():
    # Colab exports constraint files for its own package set (PIP_CONSTRAINT, UV_CONSTRAINT, ...). They must not
    # apply to the separate environment built here, which pins different versions.
    return {key: value for key, value in os.environ.items() if not key.startswith(("UV_", "PIP_"))}


if ENV_MODE == "venv":
    uv = f"{sys.executable} -m uv"
    sh(f"{sys.executable} -m pip install -q uv")
    if not os.path.exists(f"{VENV}/bin/python"):
        if sh(f"{uv} venv --python 3.11 {VENV}", env=clean_environment(), check=False) != 0:
            print("Python 3.11 could not be provisioned; building the environment on the runtime's own Python instead")
            shutil.rmtree(VENV, ignore_errors=True)
            sh(f"{uv} venv --python {sys.executable} {VENV}", env=clean_environment())
    install = f"{uv} pip install --python {VENV}/bin/python -r pyproject.toml --extra dev"
    if sh(install + " --torch-backend cu124", cwd=REPO_DIR, env=clean_environment(), check=False) != 0:
        print("retrying without --torch-backend (the default Linux wheel of torch 2.6.0 is the CUDA 12.4 build)")
        sh(install, cwd=REPO_DIR, env=clean_environment())
    PY = f"{VENV}/bin/python"
else:
    PY = sys.executable
    sh(f"{PY} -m pip install -q transformers==4.56.2 peft==0.17.1 datasets==3.6.0 accelerate==1.10.1 "
       "safetensors==0.6.2 sentencepiece==0.2.1 pytest", env=clean_environment())

sh([PY, "-c", "import torch, transformers, peft, datasets; "
              "print('torch', torch.__version__, '| cuda runtime', torch.version.cuda, '| cuda available', torch.cuda.is_available()); "
              "print('transformers', transformers.__version__, '| peft', peft.__version__, '| datasets', datasets.__version__)"])


def p0(*stages, root=P0_WORK, extra=()):
    # Run stages of experiments/p0/p0.py (baseline regression) with the training interpreter; 0 = all stages ok.
    command = [PY, "experiments/p0/p0.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda", *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def p1(*stages, root=WORK, extra=()):
    # Run stages of experiments/p1/p1.py (supply map, experiment queue, feedback); 0 = all stages ok.
    command = [PY, "experiments/p1/p1.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda",
               "--model", MODEL, "--seeds", SEEDS, *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def show_feedback(download=True):
    # Rebuild the feedback from the runs finished so far, show it, and send the bundle to the browser.
    from IPython.display import Markdown, display
    p1("feedback", extra=["--quiet"])
    report = Path(WORK) / "feedback" / "feedback_P1.md"
    if report.exists():
        display(Markdown(report.read_text()))
    if download:
        started = download_results(refresh=False)
        print("result bundle:", "download started" if started else "not downloaded")


def run_block(block, label):
    # One block of the experiment queue under the remaining budget, then the feedback so far.
    if regression_ok is False and not FORCE:
        print(f"{label} skipped: the regression check did not pass. Set FORCE = True to run it anyway.")
        STATUS[label] = "skipped (regression did not pass)"
        return
    if hours_left() < 0.3:
        print(f"{label} skipped: less than 18 minutes of budget left.")
        STATUS[label] = "skipped (no budget left)"
        return
    if regression_ok is None:
        print("NOTE: the regression check was not run in this session; the results build on an unverified baseline.")
    # Ten minutes of the budget are kept for the feedback and the download.
    done = p1("queue", extra=["--blocks", str(block), "--jobs", str(JOBS), "--deadline", str(DEADLINE - 600)]) == 0
    STATUS[label] = "all runs completed" if done else "partly done (see the queue table in the report)"
    print(f"{hours_left():.2f} h of budget left")
    show_feedback()

## B. 预演（CPU，约四分钟）

用随机初始化的小模型和合成数据，把 P0 的回归检查（上游提交、插桩提交、当前代码三方比较）和 P1 的整条流程（供给图、12 个实验臂、两个运行并行、反馈）各走一遍。它只检查流程能不能跑通，数字没有意义。失败时 notebook 停下并释放 GPU，请把输出带回。

In [ ]:
#@title B · Dry run on CPU
if RUN_TINY_DRY_RUN:
    code = sh([PY, "experiments/p0/p0.py", "env", "golden", "regress", "--tiny", "--root", TINY_P0_WORK], cwd=REPO_DIR, check=False)
    STATUS["dry run: baseline regression"] = "ok" if code == 0 else "FAILED"
    if code != 0:
        stop("The dry run of the baseline regression failed: on CPU the current code no longer matches the upstream revision. Send the output above back.")
    code = sh([PY, "experiments/p1/p1.py", "supply", "queue", "feedback", "--tiny", "--quiet", "--jobs", "2", "--root", TINY_WORK],
              cwd=REPO_DIR, check=False)
    STATUS["dry run: P1 pipeline"] = "ok" if code == 0 else "FAILED"
    if code != 0:
        stop("The dry run of the P1 pipeline failed. Nothing has been run on the GPU yet; send the output above back.")
    print(f"\ndry run ok, {hours_left():.2f} h of budget left")

## C1. 环境记录与单元测试

记录 GPU 型号、驱动和各依赖的版本，并做一次 BF16 探测；在固定版本的环境里跑一遍不需要 GPU 的测试（约一分钟）。

In [ ]:
#@title C1 · Environment record and unit tests
env_ok = p0("env") == 0
STATUS["environment"] = "ok" if env_ok else "FAILED"
if not env_ok:
    stop("Environment check failed (no CUDA device, bfloat16 not usable, or a reference revision is missing). See the warnings above.")
if RUN_TESTS:
    tests_ok = p0("tests") == 0
    STATUS["unit tests"] = "ok" if tests_ok else "FAILURES"
    print("unit tests:", "ok" if tests_ok else "FAILURES (listed above)")

## C2. 黄金值与基线回归（I1、I2）

三个冒烟配置（HO-FL、FedAvg、DeComFL，OPT-125M / SST-2，3 轮）在上游原样提交上各跑一次，在插桩提交上各跑两次，在当前代码上各跑一次，再加两次 `ho-fl-replay`。当前代码改动了 `hybrid_step` 和 `local_step`（新增默认关闭的关键字参数），这一步检查默认配置下的结果仍与上游一致。

读法：`I1 verdict` 为 `PASS_BITWISE` 或 `PASS_TOLERANCE` 即通过；`FAIL` 说明当前代码改变了基线，后面的实验组默认不再运行；`INCONCLUSIVE` 说明这块 GPU 上参照提交自己两次运行的差就超过了容差，需要研究负责人决定怎么判。

In [ ]:
#@title C2 · Golden values and regression (I1, I2)
regression_ok = None
if RUN_REGRESSION:
    golden_ok = p0("golden") == 0
    regression_ok = golden_ok and p0("regress") == 0
    STATUS["golden values"] = "ok" if golden_ok else "ATTENTION"
    STATUS["regression (I1, I2)"] = "ok" if regression_ok else "ATTENTION"
    p0("feedback", extra=["--quiet"])
    # The regression record travels with the P1 bundle.
    target = Path(WORK) / "p0_feedback"
    target.mkdir(parents=True, exist_ok=True)
    for path in [*Path(P0_WORK).glob("feedback/feedback_P0.*"), *Path(P0_WORK).glob("golden/*/golden.json"),
                 *Path(P0_WORK).glob("regress/*/verdicts.json"), Path(P0_WORK) / "env.json"]:
        if path.exists():
            shutil.copy2(path, target / (path.parent.name + "__" + path.name if path.name in ("golden.json", "verdicts.json") else path.name))
    print("\ngolden:", "ok" if golden_ok else "ATTENTION", "| regression (I1, I2):", "ok" if regression_ok else "ATTENTION")
    print(f"{hours_left():.2f} h of budget left")

## C3. E1a：供给图（不训练）

对 `dep_round_v1`（β=0 和 0.75）、`tier_equalised`、`tier_quota`（`3,1,2`、`4,1,1`、`2,2,2`）计算逐段一阶供给：期望宽度、有效来源数、最大份额，以及模拟 200 个种子乘 160 轮得到的宽度分布，旁边列出说明附录 A 的数值。只用 CPU，几秒钟。

In [ ]:
#@title C3 · E1a supply map (CPU)
supply_ok = p1("supply") == 0
STATUS["E1a supply map"] = "ok" if supply_ok else "ATTENTION"

## C4. 第 1 组：H1a（5 次运行）

`A-b0`、`C-b0`、`A-b0.75`、`Ax2-b0`、`Ax3-b0`。跑完后计算：

- G_C = C(β=0) − A(β=0)：只在一阶报告之间平均带来的提升；
- G_β = A(β=0.75) − A(β=0)：偏向采样带来的提升；
- G_lr = max(A×2, A×3) − A(β=0)：只放大步长带来的提升。

说明的判定标准：G_C ≥ 0.7 × G_β 且 G_lr < 0.5 × G_C 时支持 H1a。只有一个种子时结论标为"初步"：论文里这些运行的种子间标准差是 0.4 到 0.6 个百分点，与要比较的差值同一量级。

运行过程中每 10 轮打印一次训练损失，每次评估打印准确率。结束后显示目前为止的完整反馈并下载结果包。

In [ ]:
#@title C4 · Block 1: the runs that decide H1a
if RUN_BLOCK_1:
    run_block(1, "block 1 (H1a)")

## C5. 第 2 组：H1 与两个参照（4 次运行）

`B-b0`、`B-b0.75`（关闭零阶更新）以及 FedAvg、DeComFL。A 减 B 是零阶更新对最终准确率的净贡献；探针表给出每个客户端的零阶分量与反事实一阶分量之间的余弦，旁边是随机方向的参照值 1/√d。

In [ ]:
#@title C5 · Block 2: zeroth-order contribution (H1) and the two bounds
if RUN_BLOCK_2:
    run_block(2, "block 2 (H1, bounds)")

## C6. 第 3 组：阶梯里剩下的 3 次运行

`C-b0.75`、`D-b0`、`D-b0.75`。放不进预算的会被跳过，在报告的队列表里标为 `SKIPPED_BUDGET`。

In [ ]:
#@title C6 · Block 3: the rest of the ladder
if RUN_BLOCK_3:
    run_block(3, "block 3 (rest of the ladder)")

## D. 总反馈与曲线

汇总全部结果：回归结论、消融阶梯的准确率表（含每轮前向当量和上行字节数）、H1a 与 H1 的计算、探针表、逐段的层统计（一阶与零阶分量的范数、两两余弦、实际施加的更新的范数、所用表决集合）、供给图与各次运行实际的供给、队列与耗时、结论与下一步。同目录下还有 `summary.csv`、`curves.csv`、`layers.csv`、`probe.csv`。

In [ ]:
#@title D · Feedback report and curves
show_feedback(download=False)
STATUS["feedback"] = "written"
feedback_dir = Path(WORK) / "feedback"
data = json.loads((feedback_dir / "feedback_P1.json").read_text())
runs = data["runs"]
if runs:
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 2, figsize=(14, 4.6))
    for run in runs:
        summary = run["summary"]
        style = "--" if summary["arm"] in ("fedavg", "decomfl") else "-"
        rounds = [point["round"] for point in run["curve"]]
        axes[0].plot(rounds, [point["accuracy_pct"] for point in run["curve"]], style, marker="o", markersize=3, label=f"{summary['arm']} s{summary['seed']}")
        axes[1].plot(rounds, [point["eval_loss"] for point in run["curve"]], style, marker="o", markersize=3, label=f"{summary['arm']} s{summary['seed']}")
    axes[0].set_ylabel("test accuracy (%)")
    axes[1].set_ylabel("test loss")
    for axis in axes:
        axis.set_xlabel("round")
        axis.grid(alpha=0.3)
        axis.legend(fontsize=7, ncol=2)
    figure.suptitle(f"{data['model']} / AG News (Dirichlet 0.5)")
    figure.tight_layout()
    figure.savefig(feedback_dir / "p1_curves.png", dpi=150)
    plt.show()
else:
    print("no completed run yet")
print("report:", feedback_dir / "feedback_P1.md")
print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h")

## E. 汇总并释放 GPU

打印各步骤的状态，下载最终的结果包，然后释放 GPU 运行时（开始下载时等 30 秒，否则等 10 秒）。释放后虚拟机上的文件全部消失。

In [ ]:
#@title E · Summary, download, then release the GPU
print("stage status")
for name, state in STATUS.items():
    print(f"  {name:30s} {state}")
print("H1a:", data["h1a"]["verdict"], "| H1:", ", ".join(f"{beta} {entry['verdict']}" for beta, entry in data["h1"].items()),
      f"| {len(data['runs'])} of {data['planned_runs']} runs completed")
print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h of {BUDGET_HOURS} h")
started = download_results(refresh=True)
print("result bundle:", "download started" if started else "NOT downloaded; the report above is the record of this session")
release_gpu(30 if started else 10)

## 带回代码库的内容

- 页面上显示的 **`feedback_P1.md`**：第 3、4 节是 H1a 和 H1 的计算，第 8 节是结论和下一步。
- 浏览器下载的 **`feedback_P1_bundle.zip`**：报告、CSV 表、每次运行的 `config.json`、`summary.json`、`rounds.jsonl`、`curves.jsonl`、`ledger.jsonl`、`layers.jsonl`、`probe.jsonl`、计时和日志，以及 `p0_feedback/` 下的回归记录和黄金值。
- G1 是说明规定的停点：H1 与 H1a 的结果决定后面的设计，等研究负责人确认后再继续。

多个种子：把参数 `SEEDS` 改成 `43` 或 `43,44` 再跑一次；不同会话的结果包解压后放在一起，`python -m ho_fl.analysis <目录...> --output <目录>` 可以合并出汇总表。